# 10 · Capstone Project — Library Management System

**Goal:** combine everything from notebooks 01–09 (classes, constructors, encapsulation,
inheritance, polymorphism, abstraction, dunder methods, class/static methods) into one
realistic mini-project.

We'll build a small **Library Management System**:
- Abstract `LibraryItem` base class (abstraction)
- `Book` and `Magazine` subclasses (inheritance + polymorphism)
- Private/protected attributes with `@property` (encapsulation)
- `__str__`, `__repr__`, `__eq__`, `__lt__` (dunder methods)
- A `Library` class managing everything, with class/static methods

## Step 1 — The abstract base class

Every library item needs a title, needs to be "checked out"/"returned", and must be able to
describe itself — but *how* it describes itself differs by item type. That's abstraction +
polymorphism working together.

In [ ]:
from abc import ABC, abstractmethod

class LibraryItem(ABC):
    def __init__(self, title, item_id):
        self.title = title
        self._item_id = item_id        # protected: internal id, not meant to be reassigned casually
        self.__checked_out = False     # private: only this class's methods should flip this flag

    @property
    def checked_out(self):
        return self.__checked_out

    def check_out(self):
        if self.__checked_out:
            raise ValueError(f"'{self.title}' is already checked out.")
        self.__checked_out = True

    def return_item(self):
        if not self.__checked_out:
            raise ValueError(f"'{self.title}' was not checked out.")
        self.__checked_out = False

    @abstractmethod
    def describe(self):
        """Every subclass MUST implement its own description."""
        pass

    def __str__(self):
        status = "checked out" if self.__checked_out else "available"
        return f"{self.describe()} [{status}]"

    def __repr__(self):
        return f"{type(self).__name__}(title={self.title!r}, id={self._item_id!r})"

    def __eq__(self, other):
        return isinstance(other, LibraryItem) and self._item_id == other._item_id

    def __lt__(self, other):
        return self.title.lower() < other.title.lower()   # alphabetical ordering

## Step 2 — Concrete subclasses (inheritance + polymorphism)

Each subclass reuses everything above via `super().__init__()`, and only needs to implement
what's actually different: `describe()`.

In [ ]:
class Book(LibraryItem):
    def __init__(self, title, item_id, author, pages):
        super().__init__(title, item_id)
        self.author = author
        self.pages = pages

    def describe(self):
        return f"📖 '{self.title}' by {self.author} ({self.pages}p)"


class Magazine(LibraryItem):
    def __init__(self, title, item_id, issue_number):
        super().__init__(title, item_id)
        self.issue_number = issue_number

    def describe(self):
        return f"📰 '{self.title}' — Issue #{self.issue_number}"


# Try instantiating the abstract class directly -- should fail:
try:
    LibraryItem("Nothing", 0)
except TypeError as e:
    print("As expected, abstraction blocks this:", e)

In [ ]:
b1 = Book("Dune", "B001", "Frank Herbert", 412)
b2 = Book("1984", "B002", "George Orwell", 328)
m1 = Magazine("National Geographic", "M001", 305)

# Polymorphism: same describe()/str() call, different output per subclass
for item in [b1, b2, m1]:
    print(item)   # uses __str__, which internally calls describe() -- polymorphic dispatch

## Step 3 — Encapsulation in action

Try to check out an item twice and watch encapsulation (via the private flag + methods)
enforce the rule.

In [ ]:
b1.check_out()
print(b1)

try:
    b1.check_out()     # already checked out!
except ValueError as e:
    print("Caught:", e)

b1.return_item()
print(b1)

## Step 4 — The `Library` class (class methods + static methods + `__len__`/`__getitem__`)

This class manages a collection of items and demonstrates:
- an **instance method** (`add_item`)
- a **classmethod** used as a factory (`with_starter_catalog`)
- a **staticmethod** utility (`is_valid_id`)
- dunder methods so `Library` behaves like a container (`len()`, indexing, iteration)

In [ ]:
class Library:
    def __init__(self, name):
        self.name = name
        self.items = []

    def add_item(self, item: LibraryItem):
        if not self.is_valid_id(item._item_id):
            raise ValueError(f"Invalid item id: {item._item_id}")
        self.items.append(item)

    def find_by_title(self, title):
        for item in self.items:
            if item.title.lower() == title.lower():
                return item
        return None

    @classmethod
    def with_starter_catalog(cls, name):
        """Alternative constructor: builds a Library pre-loaded with a few items."""
        lib = cls(name)
        lib.add_item(Book("The Hobbit", "B100", "J.R.R. Tolkien", 310))
        lib.add_item(Magazine("Time", "M100", 42))
        return lib

    @staticmethod
    def is_valid_id(item_id):
        # simple utility: must be a non-empty string starting with a letter followed by digits
        return isinstance(item_id, str) and len(item_id) > 1 and item_id[0].isalpha()

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        return self.items[index]

    def __str__(self):
        return f"Library('{self.name}') with {len(self)} item(s)"

In [ ]:
lib = Library.with_starter_catalog("Downtown Library")   # built via the classmethod factory
lib.add_item(b1)
lib.add_item(b2)
lib.add_item(m1)

print(lib)
print(len(lib))          # __len__

for item in lib:          # iteration works via __getitem__
    print(" -", item)

print()
print("Sorted by title (uses __lt__):")
for item in sorted(lib.items):
    print(" -", item.title)

print()
found = lib.find_by_title("1984")
print("Found:", found)

try:
    lib.add_item(Book("Bad Book", "", "Nobody", 0))   # invalid id -> static method rejects it
except ValueError as e:
    print("Caught:", e)

## Recap: where each pillar showed up

| Pillar | Where |
|---|---|
| **Encapsulation** | `__checked_out` private flag + `check_out()`/`return_item()` controlling it; `@property checked_out` |
| **Inheritance** | `Book` and `Magazine` inherit from `LibraryItem` via `super().__init__()` |
| **Polymorphism** | `describe()` behaves differently per subclass; same `print(item)` call works for all |
| **Abstraction** | `LibraryItem(ABC)` with `@abstractmethod describe` — forces subclasses to implement it |
| **Dunder methods** | `__str__`, `__repr__`, `__eq__`, `__lt__`, `__len__`, `__getitem__` |
| **Class/static methods** | `Library.with_starter_catalog()` (classmethod factory), `Library.is_valid_id()` (staticmethod) |

### ✍️ Final challenge (extend this project yourself)

1. Add a `DVD` subclass of `LibraryItem` with a `runtime_minutes` attribute.
2. Add a `Library.overdue_items()` method (you'll need to track due dates — add a `due_date`
   attribute set when `check_out()` is called).
3. Add `__iter__`/`__next__` to `Library` to make it a proper custom iterator instead of relying
   on `__getitem__`.
4. Add a `Member` class that can hold a list of currently checked-out items, with a maximum
   limit (e.g. 5 items) enforced via encapsulation.

**You've now covered all the core OOP concepts in Python:**
classes & objects → constructors → encapsulation → inheritance → polymorphism → abstraction →
magic methods → class/static methods → and combined them in a real project. 🎉